# Notebook Task — Credit Applications by State

**Objective:** Create an application report for the state supplied as a notebook parameter.

## Why?

**For our project:** Produce state-specific credit application details from one joined dataset.

**For Data Engineering:** Reuse one notebook across For each iterations by passing the current state as a parameter.

#### 01 — Configure Notebook Parameters

Read the catalog, schema, and state parameters. Use `PE` as the default state for the initial manual test.

In [ ]:
# Define defaults for manual execution.
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "fintech_jobs")
dbutils.widgets.text("state", "PE")

# Read the current parameter values.
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
namespace = f"{catalog}.{schema}"

state = dbutils.widgets.get("state").strip().upper()
assert state in {"PE", "SP", "RJ", "MG", "BA"}, f"Unexpected state: {state}"

#### 02 — Build the State Application Report

Filter the joined applications by the supplied state. Add application calendar fields and an approval indicator, then save a separate Delta table for that state.

In [ ]:
from pyspark.sql import functions as F

# Filter applications and add reporting fields.
report_df = (
    spark.table(f"{namespace}.customers_applications_silver")
    .filter(F.col("state") == F.lit(state))
    .withColumn("application_year", F.year("application_date"))
    .withColumn("application_month", F.date_format("application_date", "MM"))
    .withColumn("is_approved", F.col("status") == F.lit("APPROVED"))
)

# Persist a separate report for the supplied state.
target_table = (
    f"{namespace}.customers_applications_{state.lower()}_silver"
)

report_df.write.format("delta").mode("overwrite").saveAsTable(
    target_table
)

# Inspect the persisted result.
stored_df = spark.table(target_table)

print(f"State: {state}")
print(f"Output table: {target_table}")
print(f"Application records: {stored_df.count()}")

display(stored_df.orderBy("application_id"))

State: PE
Output table: workspace.fintech_jobs.customers_applications_pe_silver
Application records: 2


application_id,customer_id,customer_name,state,application_date,requested_amount,status,application_year,application_month,is_approved
2001,101,Ana Ribeiro,PE,2026-10-06,3500.00,APPROVED,2026,10,true
2006,101,Ana Ribeiro,PE,2026-10-11,4500.00,PENDING,2026,10,false
